# E2E Results Analyze

End-to-end timing analysis across `(alg, dataset, method, beams, model)` combinations.

**Per-problem reduction:** within one CSV file, each problem is generated as multiple beams in parallel — so the *wall time* for that problem is the **max** `gen_time_ms` across its beams.

**Per-config aggregation:** average those per-problem max times across all problems in the dataset.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ---- Configuration space ---------------------------------------------------
ALGS     = ['best_of_n', 'beam_search']
DATASETS = ['MATH-500', 'AIME']
METHODS  = ['vllm', 'ours']
BEAMS_128    = [2, 4, 8, 16, 32, 64, 128]
BEAMS_64    = [2, 4, 8, 16, 32, 64]
MODELS   = ['1.5B', 'Llama-1B']

# ---- Paths -----------------------------------------------------------------
PROJECT_ROOT = Path('..').resolve()

# Display name -> on-disk subpath under data/. Both Qwen-1.5B and Llama-1B
# follow the same convention: data/{org}/{model_name}/{dataset_short}/.
MODEL_DIR = {
    '1.5B':     'Qwen/QWen2.5-1.5B-Instruct',
    'Llama-1B': 'meta-llama/Llama-3.2-1B-Instruct',
}

# Display name -> dataset folder under data/{MODEL_DIR}/.
# AIME's HF id is `AI-MO/aimo-validation-aime`, so the trace+replay scripts
# write under `aimo-validation-aime/` (= dataset_name.split('/')[-1]).
DATASET_DIR = {
    'MATH-500': 'MATH-500',
    'AIME':     'aimo-validation-aime',
}

def data_dir(dataset: str, model: str) -> Path:
    return PROJECT_ROOT / 'data' / MODEL_DIR[model] / DATASET_DIR[dataset]

def csv_path(alg: str, dataset: str, model: str, method: str, n: int) -> Path:
    """Filename pattern per algorithm. MATH-500 results were organized into
    per-algorithm subfolders (e.g. `.../MATH-500/best_of_n/...`); AIME results
    land flat under the dataset folder because the trace+replay scripts write
    directly to `data/{model}/{dataset_short_name}/`. Try the alg subfolder
    first, fall back to flat."""
    if alg == 'best_of_n':
        filename = f'best_of_n_request_timings_{method}_n{n}.csv'
    elif alg == 'beam_search':
        filename = f'beam_search_request_timings_{method}_n{n}.csv'
    elif alg == 'DVTS':
        filename = f'dvts_request_timings_{method}_n{n}.csv'
    else:
        raise ValueError(f'Unknown alg: {alg}')
    dir_ = data_dir(dataset, model)
    nested = dir_ / alg / filename
    flat   = dir_ / filename
    return nested if nested.exists() else flat

## 1. Per-file aggregation

For one CSV: group rows by `problem`, take `max(gen_time_ms)` per problem (the slowest beam = wall time for that problem), then average across all problems.

In [ ]:
def avg_wall_time_ms(path: Path) -> tuple[float, int]:
    """Return (mean per-problem max gen_time_ms, num_problems). NaN if file missing."""
    if not path.exists():
        return float('nan'), 0
    df = pd.read_csv(path)
    # Detect the problem column name (best_of_n uses 'problem', helper CSVs use 'problem_idx').
    problem_col = 'problem' if 'problem' in df.columns else 'problem_idx'
    per_problem_max = df.groupby(problem_col)['gen_time_ms'].max()
    return float(per_problem_max.mean()), int(per_problem_max.size)

## 2. Sweep all `(method, beams)` combinations for a given `(alg, dataset, model)`

In [ ]:
def analyze(alg: str, dataset: str, model: str) -> pd.DataFrame:
    """Build a (method x beams) table of average wall times in ms."""
    rows = []
    # 1.5B Qwen has full sweep up to n=128; Llama-1B sweep caps at n=64
    # (per LLAMA1B_SWEEP.md). Any other model defaults to BEAMS_64.
    if model == '1.5B':
        BEAMS = BEAMS_128
    else:
        BEAMS = BEAMS_64
    for method in METHODS:
        for n in BEAMS:
            p = csv_path(alg, dataset, model, method, n)
            mean_ms, n_problems = avg_wall_time_ms(p)
            rows.append({
                'method':     method,
                'beams':      n,
                'avg_ms':     mean_ms,
                'n_problems': n_problems,
                'file':       p.name if p.exists() else f'(missing) {p.name}',
            })
    long = pd.DataFrame(rows)
    pivot = long.pivot(index='method', columns='beams', values='avg_ms').reindex(METHODS)[BEAMS]
    pivot.columns = [f'n={n}' for n in pivot.columns]
    return pivot, long

## 3. Run the analysis (1.5B Model)

Pick a `(dataset, model)` and a list of algorithms to analyze. Each table shows average wall time (ms) per problem; missing files appear as `NaN`.

In [ ]:
DATASET = 'AIME'   # 'MATH-500' | 'AIME'
MODEL   = '1.5B'

print(f'Source folder: {data_dir(DATASET, MODEL)}\n')

results = {}
for alg in ALGS:
    pivot, long = analyze(alg, DATASET, MODEL)
    results[alg] = {'pivot': pivot, 'long': long}
    print(f'=== {alg} | {DATASET} | {MODEL} — avg wall time per problem (ms) ===')
    print(pivot.round(2).to_string())
    print()

## 4. Visualize best_of_n results (1.5B Model)

Latency vs beams (log-log).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import ScalarFormatter, NullFormatter

pivot = results['best_of_n']['pivot']

COLORS = {'vllm': '#80c4f5', 'ours': '#9fd564'}

x = np.arange(len(BEAMS_128))
width = 0.35

fig, ax = plt.subplots(figsize=(5, 4))
for i, method in enumerate(METHODS):
    offset = (i - (len(METHODS) - 1) / 2) * width
    vals = pivot.loc[method].values
    bars = ax.bar(x + offset, vals, width, label=method, color=COLORS[method])
    ax.bar_label(bars, fmt='%.0f', fontsize=10, padding=2)

ax.set_yscale('log')
# ax.set_ylim(3000, 15000)
# ax.set_yticks([4000, 6000, 10000, 14000])
ax.set_ylim(5000, 40000)
ax.set_yticks([10000, 20000, 30000])
_sf = ScalarFormatter()
_sf.set_scientific(False)
ax.yaxis.set_major_formatter(_sf)
ax.yaxis.set_minor_formatter(NullFormatter())
ax.set_xticks(x)
ax.tick_params(axis='both', labelsize=16)
ax.set_xticklabels([str(n) for n in BEAMS_128])
ax.set_xlabel(r'#Beams ($\mathit{b}$)', fontsize=16)
ax.set_ylabel('End-to-End Latency (ms)', fontsize=16)
ax.grid(True, axis='y', which='both', linestyle='--', alpha=0.8)
plt.tight_layout()
(PROJECT_ROOT / 'figures').mkdir(exist_ok=True)
# fig.savefig(PROJECT_ROOT / 'figures' / 'e2e_results_bon_MATH500_QWen.png',
#             dpi=300, bbox_inches='tight')
fig.savefig(PROJECT_ROOT / 'figures' / 'e2e_results_bon_AIME_QWen.png',
            dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import ScalarFormatter

pivot = results['beam_search']['pivot']

COLORS = {'vllm': '#80c4f5', 'ours': '#9fd564'}

x = np.arange(len(BEAMS_128))
width = 0.35

fig, ax = plt.subplots(figsize=(5, 4))
for i, method in enumerate(METHODS):
    offset = (i - (len(METHODS) - 1) / 2) * width
    vals = pivot.loc[method].values
    bars = ax.bar(x + offset, vals, width, label=method, color=COLORS[method])
    ax.bar_label(bars, fmt='%.0f', fontsize=10, padding=2)

ax.set_yscale('log')
# ax.set_ylim(3000, 15000)
# ax.set_yticks([4000, 6000, 10000, 14000])
ax.set_ylim(5000, 40000)
ax.set_yticks([10000, 20000, 30000])
_sf = ScalarFormatter()
_sf.set_scientific(False)
ax.yaxis.set_major_formatter(_sf)
ax.yaxis.set_minor_formatter(NullFormatter())
ax.set_xticks(x)
ax.set_xticklabels([str(n) for n in BEAMS_128])
ax.tick_params(axis='both', labelsize=16)
ax.set_xlabel(r'#Beams ($\mathit{b}$)', fontsize=16)
ax.set_ylabel('End-to-End Latency (ms)', fontsize=16)
ax.grid(True, axis='y', which='both', linestyle='--', alpha=0.8)
plt.tight_layout()
(PROJECT_ROOT / 'figures').mkdir(exist_ok=True)
# fig.savefig(PROJECT_ROOT / 'figures' / 'e2e_results_bs_MATH500_QWen.png',
#             dpi=300, bbox_inches='tight')
fig.savefig(PROJECT_ROOT / 'figures' / 'e2e_results_bs_AIME_QWen.png',
            dpi=300, bbox_inches='tight')
plt.show()

## 5. Run the analysis (Llama-1B Model)

Pick a `(dataset, model)` and a list of algorithms to analyze. Each table shows average wall time (ms) per problem; missing files appear as `NaN`.

In [ ]:
DATASET = 'AIME'   # 'MATH-500' | 'AIME'
MODEL   = 'Llama-1B'

print(f'Source folder: {data_dir(DATASET, MODEL)}\n')

results = {}
for alg in ALGS:
    pivot, long = analyze(alg, DATASET, MODEL)
    results[alg] = {'pivot': pivot, 'long': long}
    print(f'=== {alg} | {DATASET} | {MODEL} — avg wall time per problem (ms) ===')
    print(pivot.round(2).to_string())
    print()

## 6. Visualize best_of_n results (Llama-1B Model)

Latency vs beams (log-log).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import ScalarFormatter

pivot = results['best_of_n']['pivot']

COLORS = {'vllm': '#80c4f5', 'ours': '#9fd564'}

x = np.arange(len(BEAMS_64))
width = 0.35

fig, ax = plt.subplots(figsize=(5, 4))
for i, method in enumerate(METHODS):
    offset = (i - (len(METHODS) - 1) / 2) * width
    vals = pivot.loc[method].values
    bars = ax.bar(x + offset, vals, width, label=method, color=COLORS[method])
    ax.bar_label(bars, fmt='%.0f', fontsize=10, padding=2)

ax.set_yscale('log')
# ax.set_ylim(2000, 13000)
# ax.set_yticks([4000, 6000, 10000])
ax.set_ylim(4000, 28000)
ax.set_yticks([6000, 10000, 20000])
_sf = ScalarFormatter()
_sf.set_scientific(False)
ax.yaxis.set_major_formatter(_sf)
ax.yaxis.set_minor_formatter(NullFormatter())
ax.set_xticks(x)
ax.set_xticklabels([str(n) for n in BEAMS_64])
ax.tick_params(axis='both', labelsize=16)
ax.set_xlabel(r'#Beams ($\mathit{b}$)', fontsize=16)
ax.set_ylabel('End-to-End Latency (ms)', fontsize=16)
ax.grid(True, axis='y', which='both', linestyle='--', alpha=0.8)
plt.tight_layout()
(PROJECT_ROOT / 'figures').mkdir(exist_ok=True)
# fig.savefig(PROJECT_ROOT / 'figures' / 'e2e_results_bon_MATH500_Llama.png',
#             dpi=300, bbox_inches='tight')
fig.savefig(PROJECT_ROOT / 'figures' / 'e2e_results_bon_AIME_Llama.png',
            dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
pivot = results['beam_search']['pivot']

COLORS = {'vllm': '#80c4f5', 'ours': '#9fd564'}

x = np.arange(len(BEAMS_64))
width = 0.35

fig, ax = plt.subplots(figsize=(5, 4))
for i, method in enumerate(METHODS):
    offset = (i - (len(METHODS) - 1) / 2) * width
    vals = pivot.loc[method].values
    bars = ax.bar(x + offset, vals, width, label=method, color=COLORS[method])
    ax.bar_label(bars, fmt='%.0f', fontsize=10, padding=2)

ax.set_yscale('log')
# ax.set_ylim(2000, 13000)
# ax.set_yticks([4000, 6000, 10000])
ax.set_ylim(4000, 28000)
ax.set_yticks([6000, 10000, 20000])
_sf = ScalarFormatter()
_sf.set_scientific(False)
ax.yaxis.set_major_formatter(_sf)
ax.yaxis.set_minor_formatter(NullFormatter())
ax.set_xticks(x)
ax.set_xticklabels([str(n) for n in BEAMS_64])
ax.tick_params(axis='both', labelsize=16)
ax.set_xlabel(r'#Beams ($\mathit{b}$)', fontsize=16)
ax.set_ylabel('End-to-End Latency (ms)', fontsize=16)
ax.grid(True, axis='y', which='both', linestyle='--', alpha=0.8)
plt.tight_layout()
(PROJECT_ROOT / 'figures').mkdir(exist_ok=True)
# fig.savefig(PROJECT_ROOT / 'figures' / 'e2e_results_bs_MATH500_Llama.png',
#             dpi=300, bbox_inches='tight')
fig.savefig(PROJECT_ROOT / 'figures' / 'e2e_results_bs_AIME_Llama.png',
            dpi=300, bbox_inches='tight')
plt.show()